In [1]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np

from classes.GaloisField import *
from classes.GaloisPoly import *
from classes.GolayEncoder import GolayEncoder
from classes.GolayDecoder import GolayDecoder

In [2]:
gf              = GaloisField(1,0b11)
encoder_model   = GolayEncoder()
k               = encoder_model._k
n               = encoder_model._n

# compare against generated code
encoder_ouput = None
for w in range(2**k):
    w   = gf.do_unpack(w, bit_width=k)
    cw  = encoder_model.encode(w)
    encoder_ouput = cw if encoder_ouput is None else np.vstack((encoder_ouput, cw))



Field Closed Succesfully!, 1 Non-Zero Elements


In [12]:
with open("golay_code_constraint.svh", "w") as file:
    file.write("\tconstraint golay_code {\n")
    file.write("\t\trx_data inside {\n")
    for cw in encoder_ouput:
        v = gf.do_pack(cw)

        if np.all(cw == encoder_ouput[-1]):
            file.write(f"\t\t\t24'b{v:024b}\t}};\n")
        else:
            file.write(f"\t\t\t24'b{v:024b}\t,\n")
    file.write("\t};\n")

## golay 24,12 decoder model

In [3]:
r = encoder_ouput[3576]
r = r ^ np.array([
    [1,0,0,1,0,0,0,1,0,0,0,0]   ,
    [0,0,0,0,0,0,0,0,0,0,0,0]   ]).flatten()

decoder_model = GolayDecoder()

w, corrected, uncorrectable = decoder_model.decode(r)
r, decoder_model.decode(r), encoder_ouput[3576]

Field Closed Succesfully!, 1 Non-Zero Elements


(array([0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1]),
 (array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0]), True, False),
 array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1], dtype=uint8))

In [4]:
# decode all codewords, no errors
for cw in encoder_ouput:
    w, corrected, uncorrectable = decoder_model.decode(cw, full_codeword=True)
    assert(np.all(w == cw))
    assert(not corrected and not uncorrectable)

for cw in encoder_ouput:
    error_seed      = np.random.randint(0, 0b11111)
    error           = decoder_model._gf.do_unpack(error_seed, bit_width=decoder_model._n)
    error_weight    = decoder_model._gf.hamming_weight(error)

    np.random.shuffle(error)
    w, corrected, uncorrectable = decoder_model.decode(cw ^ error, full_codeword=True)

    # no errors
    if error_weight == 0:
        assert(np.all(w == cw))
        assert(not corrected and not uncorrectable)
    # 1 to 3 errors
    elif 1 <= error_weight <= 3:
        assert(np.all(w == cw))
        assert(corrected and not uncorrectable)
    # 4 errors
    else:
        assert(not np.all(w == cw))
        assert(not corrected and uncorrectable)

    # five or more errors this decoding fails, recovered bits and flags are invalid


# Test aux cells

In [15]:
s0 = decoder_model._gf.do_pack(decoder_model._G2412B[6])
s1 = decoder_model._gf.do_pack(decoder_model._G2412B[7])

hex(s0 ^ s1)

'0x783'

In [5]:
k = 12
n = 24

# define golay 24,12 matrices
G2412B = np.array([
    [1,0,0,1,1,0,0,0,1,1,1,1],
    [0,1,0,0,1,1,1,0,0,1,1,1],
    [0,0,1,1,0,1,0,1,0,1,1,1],
    [1,0,1,1,1,1,1,0,0,0,1,0],
    [1,1,0,1,1,1,0,1,0,0,0,1],
    [0,1,1,1,1,1,0,0,1,1,0,0],
    [0,1,0,1,0,0,1,1,1,1,0,1],
    [0,0,1,0,1,0,1,1,1,1,1,0],
    [1,0,0,0,0,1,1,1,1,0,1,1],
    [1,1,1,0,0,1,1,1,0,1,0,0],
    [1,1,1,1,0,0,0,1,1,0,1,0],
    [1,1,1,0,1,0,1,0,1,0,0,1]])

G2412G  = np.hstack((np.eye(k), G2412B)).astype(np.uint8)
G2412H  = np.hstack((G2412B.T, np.eye(n-k))).astype(np.uint8)


In [16]:
for r in G2412H.T:
    print(f"{decoder_model._gf.do_pack(r):03x}")

98f
4e7
357
be2
dd1
7cc
53d
2be
87b
e74
f1a
ea9
800
400
200
100
080
040
020
010
008
004
002
001


In [ ]:
``